# Primary logistic regression analysis

This notebook tests whether putting the active bias on one side changes which side participants select.

The model is

$$\operatorname{logit}(P(y_i = 1)) = \alpha + \beta X_i$$

where:

- $y_i = 1$ means participant $i$ selected Side A.
- $X_i = 0$ for baseline rows where both sides are baseline.
- $X_i = +1$ when the active bias is on Side A.
- $X_i = -1$ when the active bias is on Side B.

With this coding, $\alpha$ is the baseline log-odds of choosing Side A, and $\beta$ estimates whether the active bias pulls choices toward the side where the bias is active.

## 1. Import packages

We use `pandas` to load and clean the results file, `numpy` for simple conditional coding, and `statsmodels` for the logistic regression. The `scipy` import is only used for a one-degree-of-freedom likelihood-ratio test comparing this model to an intercept-only baseline model.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.special import expit, logit
from scipy.stats import chi2

## 2. Load the results file

The notebook can be run either from the repository root or from the `results/` folder. The path check below supports both starting locations. `results.csv` is only read; it is not modified.

In [2]:
RESULTS_PATH = Path("results.csv")

if not RESULTS_PATH.exists():
    RESULTS_PATH = Path("results/results.csv")

# The participant's choice is stored as `adjudication`; the analysis calls it `selected_side`.
raw = pd.read_csv(RESULTS_PATH).rename(columns={"adjudication": "selected_side"})
raw.head()

,dialogue_id,dialogue_word_count,case,topic,bias_type,fallacy_subtype,variant_a,variant_b,ground_truth_side_order,dialogue_variant,selected_side,reward_type
0,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,A,base_plus_incentive
1,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,B,base_reward
2,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,B,base_reward
3,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,A,base_plus_incentive
4,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,A,base_plus_incentive


## 3. Clean the three analysis columns

The input file has one row per participant/result. This analysis uses three fields:

- `variant_a`: whether Side A was `active` or `baseline`.
- `variant_b`: whether Side B was `active` or `baseline`.
- `selected_side`: which side the participant selected.

This cell standardizes capitalization and keeps only rows where the selected side is A or B.

In [3]:
df = raw.copy()
df.columns = df.columns.str.strip()

df["variant_a"] = df["variant_a"].astype(str).str.strip().str.lower()
df["variant_b"] = df["variant_b"].astype(str).str.strip().str.lower()
df["selected_side"] = df["selected_side"].astype(str).str.strip().str.upper()

df = df.loc[df["selected_side"].isin(["A", "B"])].copy()
df.shape

(369, 12)

## 4. Code the regression variables

The outcome is `y`, where `1` means Side A was selected and `0` means Side B was selected.

The predictor is `X`, a signed indicator for where the active bias appears:

- `X = 0`: baseline comparison, no active bias on either side.
- `X = 1`: active bias is on Side A.
- `X = -1`: active bias is on Side B.

The signed coding is important: if the active bias makes its side more persuasive, the model should estimate a positive `beta`, because `X = 1` should increase the probability of choosing A and `X = -1` should decrease it.

In [4]:
a_active = df["variant_a"].eq("active") & df["variant_b"].eq("baseline")
b_active = df["variant_a"].eq("baseline") & df["variant_b"].eq("active")
baseline = df["variant_a"].eq("baseline") & df["variant_b"].eq("baseline")

df["active_side"] = np.select(
    [a_active, b_active, baseline],
    ["A", "B", "baseline"],
    default="unexpected",
)

df["X"] = np.select(
    [df["active_side"].eq("A"), df["active_side"].eq("B")],
    [1, -1],
    default=0,
).astype(int)

df["y"] = df["selected_side"].eq("A").astype(int)

df[["variant_a", "variant_b", "selected_side", "active_side", "X", "y"]].head(10)

,variant_a,variant_b,selected_side,active_side,X,y
0,baseline,baseline,A,baseline,0,1
1,baseline,baseline,B,baseline,0,0
2,baseline,baseline,B,baseline,0,0
3,baseline,baseline,A,baseline,0,1
4,baseline,baseline,A,baseline,0,1
5,baseline,baseline,A,baseline,0,1
6,baseline,baseline,A,baseline,0,1
7,baseline,baseline,A,baseline,0,1
8,baseline,baseline,B,baseline,0,0
9,baseline,baseline,A,baseline,0,1


## 5. Check that the coding matches the design

Before fitting the model, this cell checks that every row is either a true baseline row or has exactly one active side. If `unexpected` appears, the regression coding would need to be revisited before interpreting the model.

In [5]:
pd.crosstab(df["active_side"], df["selected_side"], margins=True)

selected_side,A,B,All
active_side,,,
A,99,49,148
B,74,75,149
baseline,51,21,72
All,224,145,369


The model below expects no unexpected rows. This assertion intentionally stops the notebook if any row does not match the planned baseline/active-side structure.

In [6]:
assert not df["active_side"].eq("unexpected").any(), "Some rows do not match the expected design."

## 6. Look at simple descriptive rates

These descriptive rates are not the regression model, but they make the fitted model easier to sanity-check. `side_a_rate` is the observed proportion of rows in each condition where the participant selected Side A.

In [7]:
descriptives = (
    df.groupby("active_side")
    .agg(n=("y", "size"), side_a_rate=("y", "mean"))
    .reindex(["baseline", "A", "B"])
)

descriptives

,n,side_a_rate
active_side,,
baseline,72,0.708333
A,148,0.668919
B,149,0.496644


## 7. Fit the logistic regression model

`statsmodels` expects an explicit intercept column, so we add a constant to the one-column predictor dataframe. The fitted equation is exactly:

$$\operatorname{logit}(P(y_i = 1)) = \alpha + \beta X_i$$

A positive `X` coefficient means the active bias is associated with more Side A selections when placed on Side A and fewer Side A selections when placed on Side B.

In [8]:
X = sm.add_constant(df[["X"]], has_constant="add")

model = sm.Logit(df["y"], X).fit(disp=False)
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      y   No. Observations:                  369
Model:                          Logit   Df Residuals:                      367
Method:                           MLE   Df Model:                            1
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.01883
Time:                        14:04:34   Log-Likelihood:                -242.59
converged:                       True   LL-Null:                       -247.25
Covariance Type:            nonrobust   LLR p-value:                  0.002276
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.4475      0.108      4.134      0.000       0.235       0.660
X              0.3659      0.121      3.024      0.002       0.129       0.603
==============================================================================
"""

## 8. Put the model result in an easier-to-read table

The raw regression coefficient is in log-odds units. The odds ratio is often easier to read: an odds ratio above 1 means the active-side coding increases the odds of choosing Side A as `X` moves upward. Here, moving from `X = 0` to `X = 1` means moving from baseline to active-on-A; moving from `X = 0` to `X = -1` means moving from baseline to active-on-B.

In [9]:
coef_table = pd.DataFrame(
    {
        "log_odds": model.params,
        "odds_ratio": np.exp(model.params),
        "ci_low_odds_ratio": np.exp(model.conf_int()[0]),
        "ci_high_odds_ratio": np.exp(model.conf_int()[1]),
        "p_value": model.pvalues,
    }
)

coef_table

,log_odds,odds_ratio,ci_low_odds_ratio,ci_high_odds_ratio,p_value
const,0.447493,1.564385,1.265327,1.934125,0.000036
X,0.365878,1.441780,1.137387,1.827635,0.002495


## 9. Likelihood-ratio test against a baseline-only model

The coefficient table reports the usual Wald test for `beta`. As a simple model-comparison check, this cell also compares the fitted model to an intercept-only model with no `X` term. The p-value answers whether adding the active-side predictor improves model fit.

In [10]:
baseline_only_X = pd.DataFrame({"const": 1}, index=df.index)
baseline_only_model = sm.Logit(df["y"], baseline_only_X).fit(disp=False)

lr_stat = 2 * (model.llf - baseline_only_model.llf)
lr_p_value = chi2.sf(lr_stat, df=1)

pd.DataFrame(
    {
        "comparison": ["active-side model vs baseline-only model"],
        "lr_statistic": [lr_stat],
        "df": [1],
        "p_value": [lr_p_value],
    }
)

,comparison,lr_statistic,df,p_value
0,active-side model vs baseline-only model,9.312172,1,0.002276


## 10. Translate the model back into probabilities

This final table shows the model-implied probability of choosing Side A under each design condition. The active-side effect on the probability scale is summarized as half the gap between the active-on-A and active-on-B predictions. A positive value means choices move toward the side where the bias is active.

In [11]:
prediction_grid = pd.DataFrame(
    {
        "condition": ["baseline", "active on Side A", "active on Side B"],
        "const": [1, 1, 1],
        "X": [0, 1, -1],
    }
)

prediction_grid["predicted_probability_choose_A"] = model.predict(prediction_grid[["const", "X"]])

active_on_a = prediction_grid.loc[prediction_grid["X"].eq(1), "predicted_probability_choose_A"].iloc[0]
active_on_b = prediction_grid.loc[prediction_grid["X"].eq(-1), "predicted_probability_choose_A"].iloc[0]
effect_toward_active_side = 0.5 * (active_on_a - active_on_b)

prediction_grid.assign(effect_toward_active_side=effect_toward_active_side)

,condition,const,X,predicted_probability_choose_A,effect_toward_active_side
0,baseline,1,0,0.610043,0.086218
1,active on Side A,1,1,0.692827,0.086218
2,active on Side B,1,-1,0.520392,0.086218


## 11. Bootstrap confidence interval for the aggregate effect

The effect above is reported on the probability scale, so this cell computes its 95% confidence interval on the same scale. We resample rows with replacement from the final analysis dataframe, refit the same signed logistic model, recompute the same probability-shift statistic, and use the 2.5th and 97.5th percentiles of the bootstrap distribution.

In [12]:
N_BOOTSTRAP = 10000
BOOTSTRAP_SEED = 20260902
BOOTSTRAP_WARN_THRESHOLD = 0.90


def signed_logit_effect_from_arrays(y_values, x_values):
    y_values = np.asarray(y_values, dtype=float)
    x_values = np.asarray(x_values, dtype=float)

    successes = []
    totals = []
    xs = []
    for x_value in (-1.0, 0.0, 1.0):
        mask = x_values == x_value
        if np.any(mask):
            successes.append(float(y_values[mask].sum()))
            totals.append(float(mask.sum()))
            xs.append(x_value)

    successes = np.asarray(successes, dtype=float)
    totals = np.asarray(totals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    if len(xs) < 2:
        raise RuntimeError("Bootstrap sample does not contain enough X variation.")

    overall_rate = successes.sum() / totals.sum()
    if overall_rate <= 0 or overall_rate >= 1:
        raise RuntimeError("Bootstrap sample has complete outcome separation.")

    params = np.array([logit(overall_rate), 0.0], dtype=float)
    for _ in range(50):
        eta = params[0] + params[1] * xs
        p = expit(eta)
        score = np.array([
            np.sum(successes - totals * p),
            np.sum(xs * (successes - totals * p)),
        ])
        weights = totals * p * (1 - p)
        info = np.array([
            [np.sum(weights), np.sum(weights * xs)],
            [np.sum(weights * xs), np.sum(weights * xs * xs)],
        ])
        step = np.linalg.solve(info, score)
        params += step
        if np.max(np.abs(step)) < 1e-10:
            break

    if not np.all(np.isfinite(params)):
        raise RuntimeError("Bootstrap model fit returned non-finite parameters.")

    return 0.5 * (expit(params[0] + params[1]) - expit(params[0] - params[1]))


def signed_logit_effect(data):
    return signed_logit_effect_from_arrays(data["y"].to_numpy(), data["X"].to_numpy())


def bootstrap_effect_ci(data, effect_fn, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    rng = np.random.default_rng(seed)
    effects = []
    n = len(data)

    for _ in range(n_boot):
        sample_positions = rng.integers(0, n, size=n)
        sample = data.iloc[sample_positions]
        try:
            effects.append(effect_fn(sample))
        except Exception:
            continue

    effects = np.asarray(effects, dtype=float)
    if len(effects) == 0:
        raise RuntimeError("All bootstrap fits failed.")
    if len(effects) < BOOTSTRAP_WARN_THRESHOLD * n_boot:
        warnings.warn(
            f"Only {len(effects)} of {n_boot} bootstrap fits succeeded.",
            RuntimeWarning,
        )

    return {
        "estimate": effect_fn(data),
        "ci_low": np.percentile(effects, 2.5),
        "ci_high": np.percentile(effects, 97.5),
        "n_success": len(effects),
        "n_failed": n_boot - len(effects),
    }


effect_check = signed_logit_effect(df)
if not np.isclose(effect_check, effect_toward_active_side, atol=5e-7):
    raise AssertionError(
        "Bootstrap effect function does not match the notebook point estimate: "
        f"{effect_check:.8f} vs {effect_toward_active_side:.8f}"
    )

aggregate_bootstrap = bootstrap_effect_ci(df, signed_logit_effect)

aggregate_effect_pp = 100 * effect_toward_active_side
aggregate_ci_low_pp = 100 * aggregate_bootstrap["ci_low"]
aggregate_ci_high_pp = 100 * aggregate_bootstrap["ci_high"]

aggregate_effect_summary = pd.DataFrame(
    {
        "Effect (pp)": [f"{aggregate_effect_pp:+.1f}"],
        "95% bootstrap CI (pp)": [f"[{aggregate_ci_low_pp:+.1f}, {aggregate_ci_high_pp:+.1f}]"],
        "bootstrap_successes": [aggregate_bootstrap["n_success"]],
        "bootstrap_failures": [aggregate_bootstrap["n_failed"]],
    }
)

print(
    f"Aggregate effect: {aggregate_effect_pp:+.1f} pp "
    f"(95% bootstrap CI [{aggregate_ci_low_pp:+.1f}, {aggregate_ci_high_pp:+.1f}] pp)"
)
aggregate_effect_summary

Aggregate effect: +8.6 pp (95% bootstrap CI [+2.9, +14.2] pp)


,Effect (pp),95% bootstrap CI (pp),bootstrap_successes,bootstrap_failures
0,+8.6,"[+2.9, +14.2]",10000,0
